# 03｜先真正看懂 W-MSA 与 SW-MSA

这一课只解决一个问题：

> Swin 为什么先使用固定窗口做一次 Attention，然后又把窗口移动，再做一次 Attention？

先不看源码，不推复杂公式，也不讨论完整 Swin Block。我们用一个很小的方格把两个操作看明白。

## 1. 本课只需要知道的几个词

### token

图片被切成小块以后，每个小块会被转换成一组特征数字。这一组特征就是一个 token。可以先把 token 理解成“模型眼中的一个小图片区域”。

### window

window 就是窗口。它把空间上相邻的几个 tokens 分到同一组。

### Self-Attention

同一组中的 tokens 互相查看信息，再更新自己的表示。可以把它理解成一次组内交流。

### MSA

MSA 是 Multi-Head Self-Attention。Multi-Head 表示同时从多个角度进行 Self-Attention。本课只需要把它理解成“组内进行多头信息交流”。

### mask

mask 可以先理解成一张禁止交流的名单。下一小节会单独解释它怎样影响 Attention。

## 1.1 Attention Mask 到底是什么

这里的 mask 不是图像分割中覆盖物体轮廓的那种遮罩。

在 Attention 中，一个 token 会先给所有候选 tokens 打分。分数越高，表示它越想读取对方的信息。

假设 token A 正在选择要读取谁，原始分数如下：

| 候选 token | A | B | C | D |
|---|---:|---:|---:|---:|
| Attention 原始分数 | 2 | 1 | 3 | 4 |

如果没有 mask，D 的分数最高，A 会重点读取 D。

现在假设规则规定 A 不能读取 D。mask 会在 Softmax 以前，把 A→D 这个位置的分数改成一个非常小的负数：

| 候选 token | A | B | C | D |
|---|---:|---:|---:|---:|
| mask 处理后的分数 | 2 | 1 | 3 | 极小负数 |

Softmax 会把这些分数转换成总和为 1 的注意力权重。极小负数经过 Softmax 后，权重会接近 0。

结果就是：A 几乎不会读取 D 的信息。

因此，Attention Mask 的准确含义是：

> 在 Attention 分数变成权重以前，把不允许建立的连接压到接近 0。

![Attention Mask 如何屏蔽连接](images/04_attention_mask.svg)

## 1.2 mask 屏蔽的是“连接”，不是删除 token

mask 最容易被误解成把某个 token 整体删除。实际上，它通常控制的是两个 tokens 之间能不能建立 Attention 连接。

例如：

- A→D 被禁止，表示 A 不能读取 D；
- B→D 可能仍然被允许，表示 B 仍能读取 D；
- D 这个 token 本身仍然存在，也仍然可以参加其他允许的 Attention。

所以 mask 更像是在关系图上剪掉某些连线，而不是把图中的节点删除。

不同任务会使用 mask 禁止不同连接。例如语言模型可以禁止读取未来词，补齐序列时可以禁止读取无意义的填充位置。当前 Swin 学习只关心一种情况：禁止循环移位制造的虚假邻居互相读取。

## 2. 从一个 4 × 4 的 token 网格开始

假设图片已经变成下面 16 个 tokens：

| 第 1 列 | 第 2 列 | 第 3 列 | 第 4 列 |
|---|---|---|---|
| A1 | A2 | B1 | B2 |
| A3 | A4 | B3 | B4 |
| C1 | C2 | D1 | D2 |
| C3 | C4 | D3 | D4 |

现在使用 2 × 2 的窗口，因此 16 个 tokens 被分成四组：

- 窗口 A：A1、A2、A3、A4
- 窗口 B：B1、B2、B3、B4
- 窗口 C：C1、C2、C3、C4
- 窗口 D：D1、D2、D3、D4

A、B、C、D 只是为了标记四个窗口，不是类别名称。

## 3. W-MSA 是什么

W-MSA 中的 W 是 Window。

W-MSA 的完整意思是：**在固定窗口内部进行多头 Self-Attention。**

在刚才的例子中，模型分别进行四场组内交流：

- A1、A2、A3、A4 互相交流；
- B1、B2、B3、B4 互相交流；
- C1、C2、C3、C4 互相交流；
- D1、D2、D3、D4 互相交流。

四个窗口可以同时计算，但每场交流互不相通。

所以 W-MSA 可以概括为：**先固定分组，再让每组内部交换信息。**

## 4. W-MSA 以后，每个 token 得到了什么

以 A4 为例。

W-MSA 以前，A4 主要保存自己这个小区域的特征。

W-MSA 以后，A4 已经读取过 A1、A2、A3、A4 的信息，因此新的 A4 不再只代表自己，还包含窗口 A 内其他区域的信息。

B3、C2、D1 也会分别汇总自己窗口内部的信息。

但这时仍有一个明显限制：

- A4 和 B3 在图片中左右相邻，却不能交流；
- A4 和 C2 在图片中上下相邻，也不能交流；
- A、B、C、D 四组信息仍被固定窗口边界隔开。

## 5. 为什么不能每一层都使用 W-MSA

假如下一层仍然按照同样的 A、B、C、D 分组，那么 A 中的 token 还是只能查看 A，B 中的 token 还是只能查看 B。

无论重复多少次，同一个窗口内部的信息会越来越充分，但不同窗口仍然像四个互相隔离的房间。

这就是只使用 W-MSA 的问题：它解决了局部信息交流，却没有解决不同局部区域之间怎样交流。

## 6. 最关键的一步：把窗口边界移动一格

原来的窗口边界位于第 2、3 列之间，以及第 2、3 行之间。

现在把下一层的窗口边界横向和纵向各移动一格。移动以后，中间会出现一个新的 2 × 2 窗口：

| 新窗口左列 | 新窗口右列 |
|---|---|
| A4 | B3 |
| C2 | D1 |

这四个 tokens 原来分别属于 A、B、C、D 四个不同窗口。

窗口边界一移动，它们就进入了同一个新窗口，可以在下一次 Self-Attention 中互相交流。

这就是 Shifted Window 最核心的作用。

![W-MSA 与 SW-MSA 的分组变化](images/03_wmsa_swmsa.svg)

## 7. SW-MSA 是什么

SW-MSA 中的 SW 是 Shifted Window，也就是移动窗口。

SW-MSA 的完整意思是：**改变窗口边界以后，在新的窗口内部进行多头 Self-Attention。**

对于新的中间窗口，A4、B3、C2、D1 可以互相读取信息。

所以 SW-MSA 可以概括为：**改变分组，让原本属于不同窗口的 tokens 有机会交换信息。**

W-MSA 与 SW-MSA 使用的 Self-Attention 原理没有改变。改变的是哪些 tokens 被分到同一组。

## 8. 为什么先做 W-MSA，再做 SW-MSA

第一步 W-MSA：

- A4 先汇总窗口 A 的信息；
- B3 先汇总窗口 B 的信息；
- C2 先汇总窗口 C 的信息；
- D1 先汇总窗口 D 的信息。

第二步 SW-MSA：

- A4、B3、C2、D1 被放进同一个新窗口；
- 它们交流时，携带的不只是自己的原始信息，还携带各自原窗口已经汇总过的信息。

于是，一次 W-MSA 加一次 SW-MSA 就完成了两级传播：先在原窗口内部传播，再跨越原窗口边界传播。

## 9. 用一个生活化比喻理解

可以把 16 个 tokens 想成 16 名学生。

W-MSA：先按照固定座位分成 A、B、C、D 四个小组，每组内部讨论。讨论后，每个人都知道了本组其他人的观点。

SW-MSA：第二轮重新分组，让原来不同小组的部分学生坐到一起，再讨论一次。

经过两轮讨论，信息就从一个小组传播到了其他小组。

Swin 没有让全班所有人一次同时互相交流，因为那样计算量很大；它通过多轮小组讨论逐渐传播信息。

## 10. W-MSA 与 SW-MSA 到底有什么不同

| 问题 | W-MSA | SW-MSA |
|---|---|---|
| 怎样分组 | 使用固定窗口 | 移动窗口边界后重新分组 |
| 谁能互相交流 | 原窗口内部的 tokens | 来自相邻原窗口的部分 tokens |
| 主要目的 | 汇总一个局部区域内部的信息 | 让相邻局部区域交换信息 |
| Attention 原理是否改变 | 没有 | 没有 |
| token 数量是否改变 | 没有 | 没有 |
| 特征图大小是否改变 | 没有 | 没有 |

最重要的区别只有一个：**同一次 Attention 中，哪些 tokens 被分在一起。**

## 11. W-MSA 和 SW-MSA 会改变 shape 吗

不会。以 Swin-T Stage 1 为例：

| 位置 | shape |
|---|---|
| W-MSA 以前 | B × 56 × 56 × 96 |
| W-MSA 以后 | B × 56 × 56 × 96 |
| SW-MSA 以前 | B × 56 × 56 × 96 |
| SW-MSA 以后 | B × 56 × 56 × 96 |

两者都只是更新每个 token 里面的特征内容，不减少 token，也不改变通道数。

以后学习的 Patch Merging 才会让高和宽减半、通道数增加。

## 12. 概念上的移动，与实际计算中的移动

到这里，W-MSA 与 SW-MSA 的核心已经讲完。下面只解释一个实现上的麻烦。

概念上，我们只是把窗口边界移动一格。但边界移动后，图片四周会出现不完整的小窗口，不方便计算机一起处理。

实际实现会临时把整个 token 网格向左上方移动，让新的窗口重新排列成大小一致的规则方块。越过左边或上边的 tokens 会绕到另一侧，这叫循环移位。

循环移位只是让计算排列更整齐，不是 SW-MSA 的核心思想。SW-MSA 的核心始终是改变窗口分组。

## 13. 为什么循环移位还需要 mask

循环移位会产生一个副作用：原来位于图片最左侧和最右侧的 tokens，可能因为绕回操作而暂时进入同一个窗口。

但它们在真实图片中相距很远，不应该因为计算上的绕回而互相交流。

所以模型会准备一张禁止交流的名单，也就是 Attention Mask：

- 真正因为窗口边界移动而成为邻居的 tokens：允许交流；
- 只是因为循环绕回才碰到一起的 tokens：禁止交流。

mask 不负责建立跨窗口联系。跨窗口联系来自新的窗口分组；mask 只负责删除循环移位制造的错误联系。

## 14. 不要把这几个动作混在一起

| 动作 | 它真正做的事 |
|---|---|
| W-MSA | 在固定窗口内部交换信息 |
| 移动窗口边界 | 改变下一次 Attention 的分组 |
| SW-MSA | 在移动后的新窗口内部交换信息 |
| 循环移位 | 让移动后的窗口便于规则并行计算 |
| mask | 禁止循环移位制造的错误交流 |

其中最核心的是前 3 项。循环移位和 mask 是为了高效、正确地实现移动窗口。

## 15. 本节真正需要记住什么

1. W-MSA：使用固定窗口，只进行原窗口内部的信息交流。
2. 只重复 W-MSA 时，不同窗口会一直互相隔离。
3. SW-MSA：移动窗口边界后重新分组，让原来分属不同窗口的 tokens 交流。
4. W-MSA 先汇总各窗口内部的信息，SW-MSA 再把这些信息传播到相邻窗口。
5. 两者都不改变 token 数量、空间大小和通道数。
6. 循环移位是方便计算的手段，mask 用来阻止循环移位产生的错误交流。

> 一句话总结：W-MSA 负责“屋内开会”，SW-MSA 负责“重新分组后跨屋交流”。

## 16. 自测问题

1. W-MSA 中的 W 表示什么？
2. W-MSA 进行 Attention 时，窗口 A 能不能直接读取窗口 B？
3. 为什么连续使用相同窗口的 W-MSA 无法实现跨窗口交流？
4. 原例中，移动边界后为什么 A4、B3、C2、D1 可以互相交流？
5. SW-MSA 相比 W-MSA，改变的是 Attention 公式还是 token 分组？
6. 为什么先做 W-MSA，再做 SW-MSA？
7. W-MSA 和 SW-MSA 会不会减少 token 数量？
8. Attention mask 在 Softmax 以前怎样处理被禁止位置的分数？
9. mask 是删除整个 token，还是禁止某两个 tokens 之间的连接？
10. 循环移位是不是 SW-MSA 的核心目的？
11. 循环移位为什么会产生错误邻居？
12. Swin 中的 mask 负责建立跨窗口联系，还是负责删除错误联系？

### 自测参考答案

1. Window，也就是固定窗口。
2. 不能，只能读取同一个固定窗口中的 tokens。
3. 分组边界始终不变，不同窗口的 tokens 永远不会进入同一场 Attention。
4. 移动后的新窗口把它们分到了同一组。
5. 改变的是 token 分组，窗口内部仍使用相同的 Self-Attention 原理。
6. W-MSA 先汇总各原窗口内部的信息，SW-MSA 再把这些已经汇总的信息传播到相邻窗口。
7. 不会，两者只更新 token 中的特征内容。
8. 把被禁止位置的分数改成极小负数，使其经过 Softmax 后的权重接近 0。
9. 禁止某两个 tokens 之间的连接；token 本身仍然存在。
10. 不是。核心是移动窗口边界、改变分组；循环移位只是方便计算。
11. 一侧的 tokens 会绕到另一侧，使真实图片中相距很远的区域暂时碰到一起。
12. 删除循环移位制造的错误联系。真正的跨窗口联系来自移动后的新分组。